In [ ]:
# Install dependencies for this notebook (runs in the kernel)
%pip install -r ../requirements.txt

# LLM Serving Benchmark Analysis

Comparing latency under concurrent load for different model sizes.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

df = pd.read_csv('../results/results.csv')
numeric_cols = ['concurrency', 'requested', 'success_count', 'failure_count', 'avg_latency', 'min_latency', 'p50_latency', 'p95_latency', 'p99_latency', 'max_latency', 'total_tokens', 'avg_tokens', 'tokens_per_sec', 'requests_per_sec', 'duration']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['model_size'] = df['model'].apply(lambda x: 'small' if '1b' in x else 'large' if '7b' in x else x)
df['prompt_slug'] = df['prompt'].str.replace(r'[^A-Za-z0-9]+', '_', regex=True).str.strip('_')

print(f'Loaded {len(df)} rows from results.csv')
print('Prompts:', df['prompt'].nunique())
print('Models:', df['model'].nunique())
print('Devices:', df['device'].nunique())
df.head()


ModuleNotFoundError: No module named 'pandas'

In [ ]:
def style_axes(ax):
    ax.grid(True, alpha=0.3)
    ax.legend()

model_device_df = df.groupby(['model', 'device', 'concurrency'], as_index=False).agg({
    'p95_latency': 'mean',
    'avg_latency': 'mean',
    'tokens_per_sec': 'mean',
    'requests_per_sec': 'mean',
}).reset_index(drop=True)

prompt_df = df.groupby(['prompt', 'concurrency'], as_index=False).agg({
    'p95_latency': 'mean',
    'tokens_per_sec': 'mean',
}).reset_index(drop=True)

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

for (model, device), group in model_device_df.groupby(['model', 'device']):
    group = group.sort_values('concurrency')
    label = f'{model} ({device})'
    axes[0, 0].plot(group['concurrency'], group['p95_latency'], marker='s', label=label, linewidth=2)
    axes[0, 1].plot(group['concurrency'], group['avg_latency'], marker='o', label=label, linewidth=2)
    axes[1, 0].plot(group['concurrency'], group['tokens_per_sec'], marker='^', label=label, linewidth=2)
    axes[1, 1].plot(group['concurrency'], group['requests_per_sec'], marker='d', label=label, linewidth=2)

axes[0, 0].set_title('P95 Latency by Model and Device')
axes[0, 0].set_xlabel('Concurrency')
axes[0, 0].set_ylabel('P95 Latency (s)')
style_axes(axes[0, 0])

axes[0, 1].set_title('Avg Latency by Model and Device')
axes[0, 1].set_xlabel('Concurrency')
axes[0, 1].set_ylabel('Avg Latency (s)')
style_axes(axes[0, 1])

axes[1, 0].set_title('Tokens/sec by Model and Device')
axes[1, 0].set_xlabel('Concurrency')
axes[1, 0].set_ylabel('Tokens/sec')
style_axes(axes[1, 0])

axes[1, 1].set_title('Requests/sec by Model and Device')
axes[1, 1].set_xlabel('Concurrency')
axes[1, 1].set_ylabel('Requests/sec')
style_axes(axes[1, 1])

for prompt, group in prompt_df.groupby('prompt'):
    group = group.sort_values('concurrency')
    axes[2, 0].plot(group['concurrency'], group['p95_latency'], marker='s', label=prompt, linewidth=2)
axes[2, 0].set_title('P95 Latency by Prompt')
axes[2, 0].set_xlabel('Concurrency')
axes[2, 0].set_ylabel('P95 Latency (s)')
style_axes(axes[2, 0])

for prompt, group in prompt_df.groupby('prompt'):
    group = group.sort_values('concurrency')
    axes[2, 1].plot(group['concurrency'], group['tokens_per_sec'], marker='o', label=prompt, linewidth=2)
axes[2, 1].set_title('Tokens/sec by Prompt')
axes[2, 1].set_xlabel('Concurrency')
axes[2, 1].set_ylabel('Tokens/sec')
style_axes(axes[2, 1])

plt.tight_layout()
plt.show()


In [ ]:
overall = df.groupby(['model_size', 'device']).agg({'avg_latency':'mean', 'p95_latency':'mean', 'tokens_per_sec':'mean'}).reset_index()
best_latency = overall.loc[overall['avg_latency'].idxmin()]
best_throughput = overall.loc[overall['tokens_per_sec'].idxmax()]
worst_prompt = df.groupby('prompt')['p95_latency'].mean().idxmax()
highest_p95 = df.groupby('prompt')['p95_latency'].mean().max()
cpu_avg = overall.loc[overall['device'] == 'cpu', 'avg_latency'].mean()
gpu_avg = overall.loc[overall['device'] == 'gpu', 'avg_latency'].mean()
summary_text = f'''# Summary

- Best average latency: {best_latency['avg_latency']:.3f}s for {best_latency['model_size']} models on {best_latency['device']}.
- Best token throughput: {best_throughput['tokens_per_sec']:.2f} tokens/sec for {best_throughput['model_size']} models on {best_throughput['device']}.
- Highest mean p95 latency: "{worst_prompt}" at {highest_p95:.3f}s.
- GPU inference generally {'appears to' if gpu_avg < cpu_avg else 'does not appear to'} improve average latency across models (cpu={cpu_avg:.3f}, gpu={gpu_avg:.3f}).
'''
display(Markdown(summary_text))
